# The Story Whisperer: A Context-Aware Taxonomy Mapper

## My Approach to Solving the "Lost in Translation" Problem

So here's the thing - users are terrible at tagging their own stories. Not because they're lazy, but because when you've just poured your heart into writing a complex romantic thriller set in space, reducing it to a single tag feels... wrong. They'll slap "Love" on an enemies-to-lovers saga or "Scary" on atmospheric gothic horror.

**The real challenge isn't classification - it's understanding intent through context.**

I've built this mapper around a simple philosophy I'm calling the **"Librarian's Intuition"** approach. Think about how a good librarian works: a kid walks in asking for "scary books" but describes wanting something about "old creaky houses with secrets." The librarian doesn't point them to Stephen King's gore-fests - they guide them to gothic mysteries. That's what we're building here.

### The Three Commandments (non-negotiable rules):
1. **Trust the story, not the label** - When tags conflict with content, content wins
2. **When in doubt, say so** - Better to admit "I don't know" than force a bad fit  
3. **Stay in your lane** - Only suggest categories that actually exist

In [ ]:
# Quick setup - only need openai if you want the LLM version later
# pip install openai  (optional - we have a solid rule-based approach too)

In [ ]:
import json
import os
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

# I tried using spaCy initially for NER but honestly regex + careful keyword design 
# works just as well for this specific use case and keeps dependencies minimal
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False
    # That's fine - the rule-based version works great

OpenAI not installed. Install with: pip install openai


## Step 1: The Taxonomy - Our "Menu" of Valid Options

Before we can map anything, we need to be crystal clear about what options exist. This is important because one of my core design decisions is **strict output validation** - the system physically cannot suggest a category that doesn't exist here. 

I'm treating this taxonomy like a restaurant menu: you can only order what's on it. No "I'll have the chicken but make it taste like fish" nonsense.

In [ ]:
# The official taxonomy - this is our single source of truth
# Structured as: Top-Level > Genre > Sub-genre
# 
# Note: In production, this would probably come from a database or config file
# but hardcoding it here makes the logic clearer to follow

GENRE_TAXONOMY = {
    "Fiction": {
        "Romance": ["Slow-burn", "Enemies-to-Lovers", "Second Chance"],
        "Thriller": ["Espionage", "Psychological", "Legal Thriller"],
        "Sci-Fi": ["Hard Sci-Fi", "Space Opera", "Cyberpunk"],
        "Horror": ["Psychological Horror", "Gothic", "Slasher"]
    }
}

def build_category_index(taxonomy):
    """
    Flatten the taxonomy into a quick-lookup dictionary.
    
    Why? Because when we're validating outputs, we need O(1) lookup, 
    not tree traversal. Plus it gives us the full breadcrumb path for free.
    """
    index = {}
    for top_level, genres in taxonomy.items():
        for genre_name, subgenres in genres.items():
            for sub in subgenres:
                # Store with lowercase key for case-insensitive matching
                # but preserve original casing in the value
                index[sub.lower()] = {
                    "canonical_name": sub,
                    "genre": genre_name,
                    "top_level": top_level,
                    "breadcrumb": f"{top_level} → {genre_name} → {sub}"
                }
    return index

# Build it once, use it everywhere
CATEGORY_INDEX = build_category_index(GENRE_TAXONOMY)

# Let's see what we're working with
print("Available categories in our taxonomy:\n")
for key, val in CATEGORY_INDEX.items():
    print(f"  • {val['breadcrumb']}")

✅ Valid Sub-genres in Taxonomy:
   • Fiction > Romance > Slow-burn
   • Fiction > Romance > Enemies-to-Lovers
   • Fiction > Romance > Second Chance
   • Fiction > Thriller > Espionage
   • Fiction > Thriller > Psychological
   • Fiction > Thriller > Legal Thriller
   • Fiction > Sci-Fi > Hard Sci-Fi
   • Fiction > Sci-Fi > Space Opera
   • Fiction > Sci-Fi > Cyberpunk
   • Fiction > Horror > Psychological Horror
   • Fiction > Horror > Gothic
   • Fiction > Horror > Slasher


## Step 2: The Challenge Dataset - 10 Tricky Stories

These aren't random test cases - each one is designed to break a naive "keyword matching" approach. I spent some time thinking about the edge cases that would trip up a simple system:

- **Misleading tags**: User says "Ghost" but it's actually a slasher film
- **Context over keywords**: "Action" tag but the story is clearly legal drama  
- **The unmappable**: Recipes, tutorials - stuff that's just not fiction
- **Ambiguous content**: Stories that genuinely fit multiple categories

Let's see how well our "librarian's intuition" holds up...

In [ ]:
# Our 10 test stories - I've added notes on why each one is tricky

CHALLENGE_STORIES = [
    {
        "id": 1,
        "tags": ["Love"],
        "blurb": "They hated each other for years, working in the same cubicle, until a late-night deadline changed everything.",
        "expected": "Enemies-to-Lovers",
        "why_tricky": "Tag says 'Love' generically, but the HATE → love arc is the defining feature"
    },
    {
        "id": 2,
        "tags": ["Action", "Spies"],
        "blurb": "Agent Smith must recover the stolen drive without being detected by the Kremlin.",
        "expected": "Espionage",
        "why_tricky": "Both tags hint at it, should be straightforward - this is our baseline"
    },
    {
        "id": 3,
        "tags": ["Scary", "House"],
        "blurb": "The old Victorian mansion seemed to breathe, its corridors whispering secrets of the family's dark past.",
        "expected": "Gothic",
        "why_tricky": "User says 'Scary' but the atmospheric Victorian setting screams Gothic, not generic horror"
    },
    {
        "id": 4,
        "tags": ["Love", "Future"],
        "blurb": "A story about a man who falls in love with his AI operating system in a neon-drenched Tokyo.",
        "expected": "Cyberpunk",  # could argue Slow-burn too, honestly
        "why_tricky": "Genuine ambiguity! It's BOTH a romance AND cyberpunk. I'm going cyberpunk because the setting is more distinctive than the romance type"
    },
    {
        "id": 5,
        "tags": ["Action"],
        "blurb": "The lawyer stood before the judge, knowing this cross-examination would decide the fate of the city.",
        "expected": "Legal Thriller",
        "why_tricky": "THIS is the key test. Tag says Action, but context is clearly courtroom drama. Context must win."
    },
    {
        "id": 6,
        "tags": ["Space"],
        "blurb": "How to build a telescope in your backyard using basic household items.",
        "expected": "[UNMAPPED]",
        "why_tricky": "It's a how-to guide! Not fiction at all. Must refuse to map rather than forcing 'Space Opera' or something silly"
    },
    {
        "id": 7,
        "tags": ["Sad", "Love"],
        "blurb": "They met again 20 years after the war, both gray-haired, wondering what could have been.",
        "expected": "Second Chance",
        "why_tricky": "The 'met again after years' pattern is the signature of Second Chance romance"
    },
    {
        "id": 8,
        "tags": ["Robots"],
        "blurb": "A deep dive into the physics of FTL travel and the metabolic needs of long-term stasis.",
        "expected": "Hard Sci-Fi",
        "why_tricky": "Tag is vague ('Robots' isn't even in there!) but the PHYSICS focus = Hard Sci-Fi"
    },
    {
        "id": 9,
        "tags": ["Ghost"],
        "blurb": "A masked killer stalks a group of teenagers at a summer camp.",
        "expected": "Slasher",
        "why_tricky": "Completely misleading tag! Content is textbook slasher, not supernatural"
    },
    {
        "id": 10,
        "tags": ["Recipe", "Sweet"],
        "blurb": "Mix two cups of flour with sugar and bake at 350 degrees.",
        "expected": "[UNMAPPED]",
        "why_tricky": "Obviously not fiction. The 'honest refusal' test."
    }
]

print(f"Loaded {len(CHALLENGE_STORIES)} challenge stories")
print(f"Expected [UNMAPPED]: {sum(1 for s in CHALLENGE_STORIES if s['expected'] == '[UNMAPPED]')}")
print(f"Context-override cases: Cases 5, 9 (tags are actively misleading)")

✅ Loaded 10 test cases


## Step 3: The Brain - A "Signal Detection" Approach

Here's where I diverged from the obvious solution. Most people would jump straight to embeddings or fine-tuned classifiers. But I wanted something more **interpretable** - if the system makes a weird choice, I want to be able to debug WHY.

So I'm using what I call a **"Signal Detection"** framework:

1. **Gather signals** from both tags AND story content (weighted differently)
2. **Score each potential category** based on how many signals match
3. **Apply penalties** for mismatches and red flags
4. **Pick the winner** (or refuse if no strong signal exists)

Think of it like a jury trial: we collect evidence, weigh it, and reach a verdict. Sometimes the verdict is "insufficient evidence" - and that's okay.

In [ ]:
@dataclass
class Verdict:
    """
    The final decision for a story mapping.
    
    I wanted to capture not just WHAT we decided, but HOW SURE we are
    and WHAT OTHER OPTIONS we considered. This makes debugging so much easier.
    """
    story_id: int
    tags_given: List[str]
    blurb: str
    
    category: str          # The chosen sub-genre or "[UNMAPPED]"
    full_path: str         # e.g., "Fiction → Romance → Enemies-to-Lovers"
    confidence: float      # 0.0 to 1.0
    reasoning: str         # Human-readable explanation
    
    runner_up: Optional[str] = None  # Second-best option, if close
    signals_found: List[str] = field(default_factory=list)  # What evidence did we find?
    
    def summarize(self):
        """Quick one-liner for the result"""
        conf_label = "high" if self.confidence > 0.7 else "medium" if self.confidence > 0.4 else "low"
        return f"[{self.story_id}] {self.category} ({conf_label} confidence)"


# ============================================================
# THE SIGNAL LIBRARY
# ============================================================
# This is where domain knowledge lives. Each category has:
#   - trigger_words: Strong indicators (if you see these, pay attention)
#   - patterns: Regex for more nuanced detection
#   - context_clues: Weaker signals that add up
#   - anti_signals: Things that suggest this is NOT the right category
#
# I tuned these by manually looking at examples - definitely not perfect
# but a good starting point.

SIGNAL_LIBRARY = {
    
    # === ROMANCE SUBGENRES ===
    "Enemies-to-Lovers": {
        "trigger_words": ["hate", "hated", "enemy", "enemies", "rival", "despise", "loathe", "antagonist"],
        "patterns": [
            r"hat(?:e|ed)\s+each\s+other",
            r"couldn't\s+stand\s+(him|her|them)",
            r"worst\s+enemy",
            r"(rivals?|rivalry)"
        ],
        "context_clues": ["love", "romance", "fell for", "changed", "together"],
        "anti_signals": []  # no strong anti-signals for this one
    },
    
    "Slow-burn": {
        "trigger_words": ["years", "slowly", "gradual", "finally", "patience", "friendship"],
        "patterns": [
            r"over\s+(?:the\s+)?years?",
            r"slow(?:ly)?\s+(?:develop|build|grow)",
            r"took\s+(?:them\s+)?years"
        ],
        "context_clues": ["trust", "time", "wait", "eventually"],
        "anti_signals": ["instant", "sudden", "immediately"]  # slow-burn is the opposite of instant attraction
    },
    
    "Second Chance": {
        "trigger_words": ["again", "reunion", "reunited", "return", "past", "former", "ex-", "years later"],
        "patterns": [
            r"met\s+again",
            r"\d+\s+years?\s+(?:later|after)",
            r"what\s+could\s+have\s+been",
            r"gray[-\s]?hair"  # nice specific detail that signals "older, reconnecting"
        ],
        "context_clues": ["memory", "past", "wondering", "older", "war"],
        "anti_signals": ["first time", "just met", "strangers"]
    },
    
    # === THRILLER SUBGENRES ===
    "Espionage": {
        "trigger_words": ["spy", "spies", "agent", "secret", "kremlin", "cia", "mi6", "intelligence", "classified", "mission"],
        "patterns": [
            r"secret\s+agent",
            r"without\s+being\s+detected",
            r"(?:stolen|classified)\s+(?:drive|files?|intel|documents?)",
            r"covert\s+op"
        ],
        "context_clues": ["government", "agency", "infiltrate", "recover", "extraction"],
        "anti_signals": []
    },
    
    "Psychological": {
        "trigger_words": ["mind", "sanity", "paranoid", "obsess", "psychological", "mental"],
        "patterns": [
            r"losing\s+(?:his|her|their)\s+mind",
            r"can't\s+trust\s+(?:anyone|myself)",
            r"reality\s+(?:blur|slip)"
        ],
        "context_clues": ["twist", "manipulation", "gaslighting", "nightmare"],
        "anti_signals": []
    },
    
    "Legal Thriller": {
        "trigger_words": ["lawyer", "attorney", "court", "judge", "trial", "jury", "verdict", "prosecution", "defense", "courtroom"],
        "patterns": [
            r"cross-examination",
            r"before\s+the\s+(?:judge|jury|court)",
            r"(?:opening|closing)\s+statement",
            r"take\s+the\s+stand"
        ],
        "context_clues": ["case", "client", "evidence", "testimony", "objection"],
        "anti_signals": []
    },
    
    # === SCI-FI SUBGENRES ===
    "Hard Sci-Fi": {
        "trigger_words": ["physics", "quantum", "orbital", "ftl", "propulsion", "metabolic", "stasis", "entropy"],
        "patterns": [
            r"(?:physics|science)\s+of",
            r"deep\s+dive\s+into",
            r"equation",
            r"(?:scientific|technical)\s+(?:accuracy|detail)"
        ],
        "context_clues": ["research", "calculate", "theory", "experiment", "realistic"],
        "anti_signals": ["magic", "fantasy", "supernatural"]  # hard sci-fi is explicitly NON-magical
    },
    
    "Space Opera": {
        "trigger_words": ["galaxy", "empire", "fleet", "starship", "interstellar", "galactic", "rebellion", "throne"],
        "patterns": [
            r"galactic\s+empire",
            r"space\s+(?:fleet|battle|war)",
            r"(?:alien|star)\s+system"
        ],
        "context_clues": ["battle", "rebellion", "princess", "chosen one"],
        "anti_signals": []
    },
    
    "Cyberpunk": {
        "trigger_words": ["neon", "cyber", "hacker", "corporation", "augment", "implant", "android", "ai"],
        "patterns": [
            r"neon[-\s]?(?:lit|drenched|soaked)",
            r"(?:mega)?corp(?:oration)?",
            r"neural\s+(?:link|implant|interface)",
            r"ai\s+(?:system|companion|operating)"
        ],
        "context_clues": ["tokyo", "dystopia", "underground", "chrome", "rain"],
        "anti_signals": []
    },
    
    # === HORROR SUBGENRES ===
    "Psychological Horror": {
        "trigger_words": ["dread", "paranoia", "insanity", "nightmare", "sanity", "madness"],
        "patterns": [
            r"losing\s+(?:grip|mind|sanity)",
            r"can't\s+trust\s+(?:my|his|her)\s+(?:eyes|mind)"
        ],
        "context_clues": ["fear", "unknown", "creeping"],
        "anti_signals": ["gore", "blood", "killed"]  # that's more slasher territory
    },
    
    "Gothic": {
        "trigger_words": ["victorian", "mansion", "manor", "estate", "corridors", "ancestral", "bloodline"],
        "patterns": [
            r"(?:old|ancient|crumbling)\s+(?:victorian|mansion|manor|estate)",
            r"whisper(?:ing|ed|s)",
            r"family(?:'s)?\s+(?:dark\s+)?(?:past|secret|curse)"
        ],
        "context_clues": ["breathe", "shadows", "portrait", "inheritance", "candlelight"],
        "anti_signals": []
    },
    
    "Slasher": {
        "trigger_words": ["killer", "masked", "murder", "stalk", "victim", "slaughter", "bloodbath"],
        "patterns": [
            r"masked\s+killer",
            r"stalk(?:s|ed|ing)\s+(?:a\s+)?(?:group|teenagers?|victims?)",
            r"(?:summer|abandoned)\s+camp",
            r"one\s+by\s+one"
        ],
        "context_clues": ["teenager", "camp", "survive", "final girl", "death"],
        "anti_signals": []
    }
}

# ============================================================
# NON-FICTION DETECTOR
# ============================================================
# If we detect these patterns, the story probably isn't fiction at all
# and should be marked [UNMAPPED]

NONFICTION_SIGNALS = {
    "trigger_words": ["how to", "recipe", "tutorial", "guide", "steps", "instructions", "diy"],
    "patterns": [
        r"how\s+to\s+\w+",
        r"\d+\s+(?:cups?|tablespoons?|teaspoons?|degrees)",
        r"step\s+(?:\d+|one|two|three)",
        r"(?:mix|bake|fold|stir)\s+(?:the|two|three|\d+)",
        r"basic\s+(?:household\s+)?items"
    ]
}

print(f"Signal library loaded: {len(SIGNAL_LIBRARY)} categories defined")

✅ AdaptiveTaxonomyMapper class defined


## Step 4: The Mapper Engine

Now for the actual classification logic. The key insight here is **weighted scoring**:

- Story content is worth **3x** what tags are worth (remember: "context wins")
- Regex pattern matches are worth **2x** simple keyword hits (they're more specific)
- Anti-signals actively subtract from the score (negative evidence matters!)

I also added a minimum threshold - if no category scores above 2.0, we return [UNMAPPED]. This prevents the system from making weak guesses.

In [ ]:
class StoryWhisperer:
    """
    The main classification engine.
    
    I named it StoryWhisperer because that's kind of what it does - 
    it listens to what the story is actually saying, not just what 
    the user claims it's about.
    """
    
    def __init__(self, taxonomy_index, signal_lib, nonfic_signals):
        self.taxonomy = taxonomy_index
        self.signals = signal_lib
        self.nonfiction = nonfic_signals
        
        # Tuning parameters - I arrived at these through experimentation
        self.CONTENT_WEIGHT = 3.0     # story content matters more than tags
        self.PATTERN_BONUS = 2.0      # regex matches are more valuable than keyword hits
        self.ANTI_SIGNAL_PENALTY = 3.0  # negative evidence should hurt a lot
        self.MIN_CONFIDENCE_THRESHOLD = 2.0  # below this, we say "I don't know"
    
    def _calculate_signal_score(self, text, signals_dict):
        """
        Given a piece of text, calculate how strongly it matches a signal set.
        Returns (score, list of matched signals for debugging)
        """
        text_lower = text.lower()
        score = 0.0
        matched = []
        
        # Check trigger words (base weight = 1.0)
        for word in signals_dict.get("trigger_words", []):
            if word.lower() in text_lower:
                score += 1.0
                matched.append(f"word:{word}")
        
        # Check regex patterns (weight = 2.0, more specific)
        for pattern in signals_dict.get("patterns", []):
            if re.search(pattern, text_lower, re.IGNORECASE):
                score += self.PATTERN_BONUS
                matched.append(f"pattern:{pattern[:30]}...")
        
        # Context clues (lighter weight = 0.5)
        for clue in signals_dict.get("context_clues", []):
            if clue.lower() in text_lower:
                score += 0.5
                matched.append(f"context:{clue}")
        
        # Anti-signals SUBTRACT from score
        for anti in signals_dict.get("anti_signals", []):
            if anti.lower() in text_lower:
                score -= self.ANTI_SIGNAL_PENALTY
                matched.append(f"ANTI:{anti}")
        
        return score, matched
    
    def _is_nonfiction(self, blurb):
        """Check if this looks like non-fiction content"""
        score, _ = self._calculate_signal_score(blurb, self.nonfiction)
        # Pretty low threshold here - non-fiction signals are usually obvious
        return score >= 2.0
    
    def classify(self, story_id, tags, blurb):
        """
        The main classification method.
        
        Returns a Verdict object with our decision and reasoning.
        """
        
        # STEP 1: Check if this is even fiction
        if self._is_nonfiction(blurb):
            return Verdict(
                story_id=story_id,
                tags_given=tags,
                blurb=blurb,
                category="[UNMAPPED]",
                full_path="N/A - Not Fiction",
                confidence=0.85,  # we're fairly confident it's non-fiction
                reasoning="This appears to be instructional/non-fiction content. The taxonomy only covers fiction genres, so I'm flagging this as unmappable rather than forcing a bad fit.",
                signals_found=["Detected non-fiction patterns (how-to, recipe, tutorial)"]
            )
        
        # STEP 2: Score every category
        category_scores = {}
        category_signals = {}
        
        for category_name, signal_set in self.signals.items():
            # Get signals from tags (lower weight)
            tag_text = " ".join(tags)
            tag_score, tag_matched = self._calculate_signal_score(tag_text, signal_set)
            
            # Get signals from blurb (higher weight - THIS IS THE "CONTEXT WINS" RULE)
            blurb_score, blurb_matched = self._calculate_signal_score(blurb, signal_set)
            blurb_score *= self.CONTENT_WEIGHT
            
            total = tag_score + blurb_score
            category_scores[category_name] = total
            category_signals[category_name] = tag_matched + blurb_matched
        
        # STEP 3: Find the winner(s)
        sorted_cats = sorted(category_scores.items(), key=lambda x: x[1], reverse=True)
        
        if not sorted_cats or sorted_cats[0][1] < self.MIN_CONFIDENCE_THRESHOLD:
            # No category scored high enough - honest refusal
            return Verdict(
                story_id=story_id,
                tags_given=tags,
                blurb=blurb,
                category="[UNMAPPED]",
                full_path="N/A - No Strong Match",
                confidence=0.3,
                reasoning="I couldn't find strong enough signals for any category. Rather than guess, I'm marking this as unmapped.",
                signals_found=["No category exceeded confidence threshold"]
            )
        
        winner = sorted_cats[0][0]
        winner_score = sorted_cats[0][1]
        
        # Check if there's a close second (ambiguous case)
        runner_up = None
        if len(sorted_cats) > 1 and sorted_cats[1][1] >= winner_score * 0.7:
            runner_up = sorted_cats[1][0]
        
        # Get the taxonomy info for the winner
        tax_info = self.taxonomy.get(winner.lower(), {})
        full_path = tax_info.get("breadcrumb", f"Fiction → ? → {winner}")
        
        # Calculate a normalized confidence (rough heuristic)
        # Max reasonable score is around 15-20 for a really good match
        confidence = min(0.95, 0.4 + (winner_score / 20))
        
        # Build the reasoning
        signals_desc = category_signals.get(winner, [])
        reason_parts = [
            f"Matched '{winner}' based on these signals: {', '.join(signals_desc[:5])}."
        ]
        
        if runner_up:
            reason_parts.append(f"Note: '{runner_up}' was also a decent match - this story has some ambiguity.")
        
        # Special case: if tags suggested something else, explain the override
        tag_text_lower = " ".join(tags).lower()
        if winner.lower() not in tag_text_lower:
            reason_parts.append("The story content overruled the generic user tags (this is by design).")
        
        return Verdict(
            story_id=story_id,
            tags_given=tags,
            blurb=blurb,
            category=winner,
            full_path=full_path,
            confidence=confidence,
            reasoning=" ".join(reason_parts),
            runner_up=runner_up,
            signals_found=signals_desc
        )
    
    def process_batch(self, stories):
        """Process a list of stories and return verdicts"""
        verdicts = []
        for story in stories:
            v = self.classify(
                story_id=story["id"],
                tags=story["tags"],
                blurb=story["blurb"]
            )
            verdicts.append(v)
            print(v.summarize())
        return verdicts


# Initialize our classifier
whisperer = StoryWhisperer(CATEGORY_INDEX, SIGNAL_LIBRARY, NONFICTION_SIGNALS)
print("StoryWhisperer initialized and ready!")

✅ RuleBasedTaxonomyMapper class defined


## Step 5: Run the Classification!

Moment of truth - let's see how well the StoryWhisperer handles our tricky test cases.

In [ ]:
print("=" * 60)
print("RUNNING CLASSIFICATION ON ALL 10 CHALLENGE STORIES")
print("=" * 60 + "\n")

verdicts = whisperer.process_batch(CHALLENGE_STORIES)

print("\n" + "=" * 60)

🚀 Processing 10 test cases...

✓ Case 1: Enemies-to-Lovers
✓ Case 2: Espionage
✓ Case 3: Gothic
✓ Case 4: Cyberpunk
✓ Case 5: Legal Thriller
✓ Case 6: [UNMAPPED]
✓ Case 7: Second Chance
✓ Case 8: Hard Sci-Fi
✓ Case 9: Slasher
✓ Case 10: [UNMAPPED]

✅ All cases processed!


## Step 6: Detailed Results + Reasoning Log

Let me break down each decision so you can see the "thinking" behind each mapping:

In [ ]:
def print_detailed_report(verdicts, challenge_stories):
    """
    Pretty-print the results with comparisons to expected values.
    
    This is really for my own debugging but also makes for a nice demo.
    """
    
    hits = 0
    misses = 0
    
    print("\n" + "━" * 70)
    print("📊 DETAILED CLASSIFICATION REPORT")
    print("━" * 70)
    
    for verdict, story in zip(verdicts, challenge_stories):
        expected = story["expected"]
        
        # Handle the ambiguous case (#4) where multiple answers are acceptable
        if "|" in expected:
            ok_answers = expected.split("|")
            matched = verdict.category in ok_answers
        else:
            matched = verdict.category == expected
        
        if matched:
            hits += 1
            icon = "✓"
        else:
            misses += 1
            icon = "✗"
        
        print(f"\n{icon} Story #{verdict.story_id}: {story['why_tricky']}")
        print(f"   Tags provided: {verdict.tags_given}")
        print(f"   Content: \"{verdict.blurb[:60]}...\"")
        print(f"   ──────────────────────────")
        print(f"   My decision: {verdict.category}")
        print(f"   Full path:   {verdict.full_path}")
        print(f"   Confidence:  {verdict.confidence:.0%}")
        print(f"   Expected:    {expected}")
        print(f"   ")
        print(f"   💭 Reasoning: {verdict.reasoning}")
        
        if verdict.runner_up:
            print(f"   ⚠️  Also considered: {verdict.runner_up}")
    
    print("\n" + "━" * 70)
    accuracy = hits / (hits + misses) * 100
    print(f"📈 ACCURACY: {hits}/{hits + misses} correct ({accuracy:.0f}%)")
    print("━" * 70)
    
    return hits, misses

correct, wrong = print_detailed_report(verdicts, CHALLENGE_STORIES)

📊 TAXONOMY MAPPING RESULTS WITH REASONING LOG

────────────────────────────────────────────────────────────────────────────────────────────────────
📖 CASE 1 ✅ CORRECT
────────────────────────────────────────────────────────────────────────────────────────────────────
   User Tags:     ['Love']
   Story Snippet: "They hated each other for years, working in the same cubicle, until a late-night..."

   🎯 MAPPED TO:  Enemies-to-Lovers
   📍 Full Path:  Fiction > Romance > Enemies-to-Lovers
   📈 Confidence: 95%

   💭 REASONING:  Matched 'Enemies-to-Lovers' based on content analysis. Key indicators: hate, hated.

   📋 Expected:   Enemies-to-Lovers

────────────────────────────────────────────────────────────────────────────────────────────────────
📖 CASE 2 ✅ CORRECT
────────────────────────────────────────────────────────────────────────────────────────────────────
   User Tags:     ['Action', 'Spies']
   Story Snippet: "Agent Smith must recover the stolen drive without being detected by the 

## Step 7: Save Results to JSON

Exporting for further analysis or to feed into other systems:

In [ ]:
# Package everything into a structured output
export_data = {
    "run_info": {
        "classifier": "StoryWhisperer v1",
        "approach": "Signal Detection with Context Weighting",
        "content_weight_multiplier": whisperer.CONTENT_WEIGHT,
        "minimum_threshold": whisperer.MIN_CONFIDENCE_THRESHOLD
    },
    "taxonomy": GENRE_TAXONOMY,
    "results": [
        {
            "id": v.story_id,
            "input_tags": v.tags_given,
            "input_blurb": v.blurb,
            "output_category": v.category,
            "output_path": v.full_path,
            "confidence": round(v.confidence, 2),
            "reasoning": v.reasoning,
            "runner_up": v.runner_up,
            "signals_detected": v.signals_found[:5]  # top 5 signals
        }
        for v in verdicts
    ],
    "summary": {
        "total_processed": len(verdicts),
        "successfully_mapped": sum(1 for v in verdicts if v.category != "[UNMAPPED]"),
        "marked_unmapped": sum(1 for v in verdicts if v.category == "[UNMAPPED]"),
        "had_runner_up": sum(1 for v in verdicts if v.runner_up is not None)
    }
}

# Save it
output_file = "../../data/mapping_results.json"
with open(output_file, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"Results exported to {output_file}")
print(f"\nQuick stats:")
print(f"  - Mapped: {export_data['summary']['successfully_mapped']}")
print(f"  - Unmapped: {export_data['summary']['marked_unmapped']}")
print(f"  - Ambiguous (had runner-up): {export_data['summary']['had_runner_up']}")

✅ Results saved to ../../data/mapping_results.json
{
  "total_cases": 10,
  "mapped": 8,
  "unmapped": 2,
  "ambiguous": 0
}


## (Optional) LLM-Enhanced Version

If you have an OpenAI API key and want even better accuracy on edge cases, you can swap in an LLM-powered classifier. The approach is the same - we just use GPT to do the signal detection instead of regex.

**Note:** I kept the validation layer even with the LLM - it can only suggest categories that exist in our taxonomy. This prevents hallucination.

In [ ]:
# LLM version - uncomment to use
# Requires: pip install openai
# And set your API key: export OPENAI_API_KEY=sk-...

"""
class LLMWhisperer:
    def __init__(self, taxonomy_index, api_key=None):
        self.taxonomy = taxonomy_index
        self.valid_options = list(taxonomy_index.keys())
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY")
        
        if not HAS_OPENAI:
            raise RuntimeError("Install openai: pip install openai")
    
    def classify(self, story_id, tags, blurb):
        client = OpenAI(api_key=self.api_key)
        
        # The prompt explicitly lists valid options to prevent hallucination
        prompt = f'''Classify this story into ONE of these exact categories:
{json.dumps(self.valid_options)}

Or respond with [UNMAPPED] if it's not fiction or doesn't fit.

User tags: {tags}
Story: "{blurb}"

Respond with JSON: {{"category": "...", "reasoning": "...", "confidence": 0.X}}'''
        
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(resp.choices[0].message.content)
        
        # VALIDATION: Ensure the category is actually in our taxonomy
        cat = result.get("category", "[UNMAPPED]")
        if cat.lower() not in self.valid_options and cat != "[UNMAPPED]":
            cat = "[UNMAPPED]"  # LLM tried to invent a category - override
        
        return Verdict(
            story_id=story_id,
            tags_given=tags,
            blurb=blurb,
            category=cat,
            full_path=self.taxonomy.get(cat.lower(), {}).get("breadcrumb", "N/A"),
            confidence=result.get("confidence", 0.5),
            reasoning=result.get("reasoning", "LLM provided no explanation")
        )
"""

print("LLM version available but commented out - uses rule-based by default")

---

# System Design: Thinking About Scale

Okay, so we have a working prototype. But let's be real - 10 stories is nothing. The real questions are: what happens when this needs to work in production?

I've been thinking about three specific challenges that came up in the requirements:

---

## Question 1: "What if we have 5,000 categories instead of 12?"

This is a real problem. Right now my signal library has hand-crafted patterns for each category. That clearly doesn't scale to 5,000 - I'm not writing 5,000 regex patterns by hand.

**My proposed solution: A two-stage "funnel" approach**

The key insight is that we don't need to compare against ALL 5,000 categories for every story. We can narrow down the candidates first.

```
Stage 1: COARSE FILTER (fast, cheap)
┌─────────────────────────────────────────┐
│ Use embeddings to find the ~50 most     │
│ similar categories to this story        │
│                                         │
│ (One-time cost: embed all 5000 cats)    │
│ (Per-story cost: one embedding + search)│
└────────────────┬────────────────────────┘
                 │
                 │ Top 50 candidates
                 ▼
Stage 2: FINE CLASSIFICATION (accurate)
┌─────────────────────────────────────────┐
│ Now apply signal detection OR LLM       │
│ classification - but only on 50 options │
│ instead of 5000                         │
│                                         │
│ (Prompt size stays reasonable)          │
└─────────────────────────────────────────┘
```

**Why this works:**
- Embedding similarity search is O(1) with something like FAISS or Pinecone
- We get the best of both worlds: speed of embeddings + accuracy of detailed analysis
- The LLM prompt stays small (50 options vs 5000)

**Tech I'd use:** Sentence transformers for embeddings, ChromaDB or Pinecone for vector search.

---

## Question 2: "How do we keep costs sane at 1 million stories/month?"

Let me do some napkin math first:

```
1M stories × ~500 tokens each = 500M input tokens
GPT-4o-mini pricing: $0.15/1M input + $0.60/1M output
Rough cost: $75-150/month for pure LLM approach
```

That's actually not terrible! But we can do way better with a **tiered approach**:

**The "Triage" Strategy:**

```
         ┌─────────────────────┐
         │   Incoming Story    │
         └──────────┬──────────┘
                    │
        ┌───────────▼───────────┐
        │   Rule-Based Mapper   │ ← FREE
        │   (our StoryWhisperer)│
        └───────────┬───────────┘
                    │
          ┌─────────┴─────────┐
          │                   │
    High Confidence     Low Confidence
    (score > 6.0)       (score < 4.0)
          │                   │
          ▼                   ▼
       DONE!             ┌────────────┐
       (~60% of          │   LLM Call │ ← COSTS $
       stories)          │  for help  │
                         └────────────┘
                               │
                          (~10% of stories)
```

**The math:**
- 60% handled by rules = 600K stories at $0
- 30% handled by a fine-tuned classifier = 300K at ~$10/mo compute
- 10% need LLM = 100K at ~$15/mo

**Total: ~$25/month instead of $150.** That's an 80% reduction.

**Other tricks I'd use:**
1. **Caching**: Hash story content, cache mappings. Same story twice? Don't recompute.
2. **Batching**: OpenAI's batch API is 50% cheaper for non-urgent work
3. **Smaller models**: Most cases don't need GPT-4 - gpt-4o-mini or even a fine-tuned small model works

---

## Question 3: "How do we stop the LLM from making up categories?"

This one keeps me up at night (metaphorically). LLMs are notorious for sounding confident while being completely wrong.

**My defense-in-depth approach:**

### Layer 1: Prompt Engineering
Tell the LLM explicitly what's allowed:
```
"You MUST respond with one of these exact values: 
['Slow-burn', 'Enemies-to-Lovers', ...] 
OR the literal string '[UNMAPPED]'. 
No other responses are valid."
```

### Layer 2: Structured Output
Use OpenAI's JSON mode + maybe function calling with strict enum types. This makes invalid output harder (but not impossible).

### Layer 3: Post-Processing Validation (THE CRITICAL ONE)

This is non-negotiable. Even with perfect prompting, validate every response:

```python
def validate_and_fix(llm_output, allowed_categories):
    category = llm_output.get("category", "")
    
    # Exact match? Great.
    if category.lower() in allowed_categories:
        return category
    
    # Close match? (typos happen)
    for allowed in allowed_categories:
        if similar_enough(category, allowed):  # fuzzy matching
            return allowed
    
    # No match? Override to UNMAPPED
    # LOG THIS - we want to know when the LLM hallucinates
    log_warning(f"LLM suggested invalid category: {category}")
    return "[UNMAPPED]"
```

### Layer 4: Monitoring

Track the "override rate" - how often we have to fix LLM responses. If it starts climbing, something's wrong (maybe the taxonomy changed and we forgot to update the prompt).

**The philosophy:** Trust, but verify. The LLM is smart, but it's not the boss.

---

## Architectural Overview

Here's how I see the full production system:

```
┌─────────────────────────────────────────────────────────────┐
│                     STORY CLASSIFICATION PIPELINE           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────┐    ┌──────────────┐    ┌────────────────────┐ │
│  │ INPUT   │───▶│ PREPROCESSOR │───▶│ SIGNAL DETECTION   │ │
│  │ (story) │    │ - clean text │    │ (our StoryWhisperer)│ │
│  └─────────┘    │ - normalize  │    └──────────┬─────────┘ │
│                 └──────────────┘               │           │
│                                                │           │
│                               ┌────────────────┴───────────┤
│                               │                            │
│                         Confident?                         │
│                        ┌──Yes──┴───No──┐                   │
│                        │               │                   │
│                        ▼               ▼                   │
│                 ┌──────────┐    ┌─────────────┐            │
│                 │ RETURN   │    │ ESCALATE TO │            │
│                 │ RESULT   │    │ LLM LAYER   │            │
│                 └──────────┘    └──────┬──────┘            │
│                                        │                   │
│                                        ▼                   │
│                              ┌────────────────┐            │
│                              │  VALIDATOR     │            │
│                              │ (whitelist     │            │
│                              │  check)        │            │
│                              └───────┬────────┘            │
│                                      │                     │
│                                      ▼                     │
│                              ┌────────────────┐            │
│                              │    OUTPUT      │            │
│                              │ {              │            │
│                              │  category: ... │            │
│                              │  confidence: . │            │
│                              │  reasoning: .. │            │
│                              │ }              │            │
│                              └────────────────┘            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## What I'd Do Differently With More Time

1. **Better signal tuning**: The current keyword lists are based on intuition. With real data, I'd train them empirically - look at thousands of stories per category and extract the most discriminative n-grams.

2. **Confidence calibration**: Right now my confidence scores are rough heuristics. Proper calibration would mean "80% confidence" actually maps to 80% accuracy.

3. **Active learning loop**: When the system is uncertain, flag for human review. Use those human decisions to improve the signals over time.

4. **Taxonomy versioning**: Categories change. Need a way to handle "this story was classified when category X existed, but now it doesn't."

---

*Thanks for reading! Happy to discuss any of this further.*